# Unsupervised Soil Viability Recommendation System
This notebook builds an end-to-end unsupervised machine learning pipeline to analyze unlabeled soil telemetry data. 
It clusters the data into separate soil profiles, uses agricultural heuristics to automatically classify the clusters into **Viable** vs. **Unviable** categories, and builds a production-ready live recommendation engine.

## 1. Environment Setup & Dependency Imports
*Loads robust packages for numerical computing, feature scaling, K-Means clustering, Dimensionality Reduction (PCA), metric reporting, and visual plotting.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Configure global visualization settings
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14

print("[+] Environment and charting modules successfully configured.")

## 2. Exploratory Data Analysis (EDA)
*Loads the downsampled 5% dataset, runs integrity audits for missing parameters, analyzes statistical dispersion, and maps interactions via a correlation heatmap.*

In [ ]:
# Load the 5% downsampled soil dataset
data_path = "Soil_Dataset_5percent_Sample.csv"
df = pd.read_csv(data_path)
print(f"[+] Dataset Loaded. Total Sample Rows: {df.shape[0]} | Columns: {df.shape[1]}\n")

# Isolate our target soil chemical features (ignoring spatial coordinate markers for ML)
features = ["Nitrogen", "Phosphorus", "Potassium", "pH", "EC"]
X_raw = df[features]

print("--- Structural Check & Missing Value Count ---")
print(X_raw.info())
print(f"\nMissing cells per feature:\n{X_raw.isnull().sum()}")

print("\n--- Descriptive Statistical Distribution ---")
print(X_raw.describe().T)

# Plotting Feature Distribution Histograms
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, col in enumerate(features):
    sns.histplot(X_raw[col], kde=True, ax=axes[i], color="teal")
    axes[i].set_title(f"{col} Distribution")
plt.tight_layout()
plt.show()

# Multi-Feature Correlation Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(X_raw.corr(), annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Soil Chemical Feature Interactions (Correlation)")
plt.tight_layout()
plt.show()

## 3. Feature Normalization & Scaling
*Because macros like Nitrogen or Potassium span wide numeric bounds (0–200+) while trace traits like pH sit strictly between 0–14, we apply a standard Z-score transformation (`StandardScaler`). This ensures all features contribute equally to distance calculations.*

In [ ]:
print("[*] Scaling soil attributes to uniform variance...")

# Initialize and fit the Z-score normalizer
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Convert back to a DataFrame briefly to verify scaling results
X_scaled_df = pd.DataFrame(X_scaled, columns=features)
print("\n--- Scaled Features Verification (Expect Mean ≈ 0, StDev ≈ 1) ---")
print(X_scaled_df.describe().T[["mean", "std"]])

## 4. Unsupervised K-Means Machine Learning Training
*Trains a K-Means algorithm optimized to cluster the dataset into exactly two distinct operational categories: stable, highly productive zones versus deficient, hazardous zones.*

In [ ]:
print("[*] Initiating K-Means clustering algorithm...")

# Instantiating K-Means with k=2 clusters, utilizing k-means++ for optimal initial seed center choices
kmeans = KMeans(n_clusters=2, init="k-means++", random_state=42, n_init=10)
df["Cluster_ID"] = kmeans.fit_predict(X_scaled)

print("\n[+] Clustering complete. Row allocation per generated segment:")
cluster_counts = df["Cluster_ID"].value_counts()
for cid, count in cluster_counts.items():
    print(f" -> Cluster {cid}: {count} rows ({count/len(df)*100:.2f}%)")

## 5. Domain Knowledge Alignment (Automated Cluster Labeling)
*Unsupervised models only group vectors; they do not know what 'fertile' means. This cell calculates cluster feature means and automatically determines which cluster represents standard, healthy soil criteria (e.g., pH near an ideal target of 6.5).*

In [ ]:
# 1. Calculate global benchmarks from the dataset to normalize our scoring
global_means = df[features].mean()
global_stds = df[features].std()

# 2. Compute the real-world feature averages for each mathematical cluster
cluster_averages = df.groupby("Cluster_ID")[features].mean()
print("--- Mean Chemical Footprint per Cluster ---")
print(cluster_averages)
print("\n" + "="*50 + "\n")

# 3. Apply Multi-Criteria Soil Quality Scoring Heuristic
cluster_scores = {}

for cluster_id, row in cluster_averages.iterrows():
    print(f"Calculating Agronomic Health Score for Cluster {cluster_id}:")
    
    # Nutrient Score: Z-score relative to global dataset (Higher NPK = Better)
    n_score = (row["Nitrogen"] - global_means["Nitrogen"]) / global_stds["Nitrogen"]
    p_score = (row["Phosphorus"] - global_means["Phosphorus"]) / global_stds["Phosphorus"]
    k_score = (row["Potassium"] - global_means["Potassium"]) / global_stds["Potassium"]
    total_nutrient_score = n_score + p_score + k_score
    print(f" -> Relative Nutrient Score (NPK): {total_nutrient_score:+.4f}")
    
    # pH Penalty: Absolute deviation from ideal neutral crop growth (6.5)
    ph_penalty = abs(row["pH"] - 6.5)
    print(f" -> pH Divergence Penalty: {ph_penalty:.4f}")
    
    # Salinity (EC) Penalty: High EC indicates high salinity stress (Higher EC = Worse)
    ec_penalty = (row["EC"] - global_means["EC"]) / global_stds["EC"]
    print(f" -> Relative Salinity Stress Penalty (EC): {ec_penalty:+.4f}")
    
    # Final Formula: Nutrients - pH Penalty - Salinity Penalty
    final_health_score = total_nutrient_score - ph_penalty - ec_penalty
    cluster_scores[cluster_id] = final_health_score
    print(f" ==> TOTAL CLUSTER HEALTH SCORE: {final_health_score:.4f}\n")

# 4. Programmatically assign recommendations based on the highest score
viable_cluster = max(cluster_scores, key=cluster_scores.get)
unviable_cluster = 1 if viable_cluster == 0 else 0

recommendation_map = {
    viable_cluster: "Viable / Recommended",
    unviable_cluster: "Unviable / Deficient"
}

print("="*50)
print(f"[+] Automated Heuristic Mapping Decision:")
print(f" -> Cluster {viable_cluster} selected as: {recommendation_map[viable_cluster]} (Highest Health Score)")
print(f" -> Cluster {unviable_cluster} selected as: {recommendation_map[unviable_cluster]}")
print("="*50)

# Map labels back to the master DataFrame for downstream visualization evaluation
df["System_Recommendation"] = df["Cluster_ID"].map(recommendation_map)

## 6. Advanced Evaluation & Diagnostic Dashboard
*Evaluates the model's structural soundness. Since we have 5 dimensions, we compress them into a 2D map via Principal Component Analysis (PCA) to check segment boundaries, then inspect nutrient variance using boxplots.*

In [ ]:
print("[*] Compiling 2D projection and 5-feature profile diagnostics (Log-Scaled Metrics)...")

# 1. Initialize PCA to compress 5 features into 2 principal dimensions for geometric visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Store components in DataFrame for plotting
df["PCA_1"] = X_pca[:, 0]
df["PCA_2"] = X_pca[:, 1]

# 2. Setup a 2x3 multi-panel figure grid to accommodate PCA + all 5 agricultural features
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Explicitly map colors to recommendations so they remain uniform across all subplots
recommendation_palette = {
    "Viable / Recommended": "forestgreen", 
    "Unviable / Deficient": "crimson"
}

# --- Subplot 1: PCA 2D Cluster Space Separation ---
# NOTE: Kept linear because PCA contains negative coordinates; log scaling here would break the rendering.
sns.scatterplot(
    data=df, x="PCA_1", y="PCA_2", hue="System_Recommendation",
    palette=recommendation_palette, alpha=0.5, s=12, ax=axes[0, 0]
)
axes[0, 0].set_title("Soil Cluster Boundaries (PCA Projection)")
axes[0, 0].set_xlabel("Principal Component 1")
axes[0, 0].set_ylabel("Principal Component 2")
axes[0, 0].legend(loc="upper right") # Bypasses slow dynamic "loc=best" calculation

# --- Subplot 2: Nitrogen Profiles (Log Scale) ---
sns.boxplot(
    data=df, x="System_Recommendation", y="Nitrogen", 
    hue="System_Recommendation", palette=recommendation_palette, 
    legend=False, ax=axes[0, 1]
)
axes[0, 1].set_yscale("log")  # Applies log transform to Y-axis
axes[0, 1].set_title("Nitrogen Distribution (Log Scale)")
axes[0, 1].set_xlabel("")

# --- Subplot 3: Phosphorus Profiles (Log Scale) ---
sns.boxplot(
    data=df, x="System_Recommendation", y="Phosphorus", 
    hue="System_Recommendation", palette=recommendation_palette, 
    legend=False, ax=axes[0, 2]
)
axes[0, 2].set_yscale("log")  # Applies log transform to Y-axis
axes[0, 2].set_title("Phosphorus Distribution (Log Scale)")
axes[0, 2].set_xlabel("")

# --- Subplot 4: Potassium Profiles (Log Scale) ---
sns.boxplot(
    data=df, x="System_Recommendation", y="Potassium", 
    hue="System_Recommendation", palette=recommendation_palette, 
    legend=False, ax=axes[1, 0]
)
axes[1, 0].set_yscale("log")  # Applies log transform to Y-axis
axes[1, 0].set_title("Potassium Distribution (Log Scale)")
axes[1, 0].set_xlabel("")

# --- Subplot 5: pH Profiles (Log Scale) ---
sns.boxplot(
    data=df, x="System_Recommendation", y="pH", 
    hue="System_Recommendation", palette=recommendation_palette, 
    legend=False, ax=axes[1, 1]
)
axes[1, 1].set_yscale("log")  # Applies log transform to Y-axis
axes[1, 1].set_title("pH Distribution (Log Scale)")
axes[1, 1].set_xlabel("")

# --- Subplot 6: Electrical Conductivity (EC / Salinity) Profiles (Log Scale) ---
sns.boxplot(
    data=df, x="System_Recommendation", y="EC", 
    hue="System_Recommendation", palette=recommendation_palette, 
    legend=False, ax=axes[1, 2]
)
axes[1, 2].set_yscale("log")  # Applies log transform to Y-axis
axes[1, 2].set_title("Electrical Conductivity (Log Scale)")
axes[1, 2].set_xlabel("")

plt.tight_layout()
plt.show()

## 7. Pipeline Asset Persistence & Inference Testing
*Exports the fitted scaler weights, the trained K-Means center metrics, and the mapped label dictionary as a single `.pkl` file. Then, sets up a modular prediction engine for live data point checks.*

In [ ]:
# Bundle pipeline artifacts together
soil_model_payload = {
    "scaler_transform": scaler,
    "clustering_engine": kmeans,
    "label_map": recommendation_map
}

# Persist deployment package to local file storage
model_filename = "unsupervised_soil_pipeline.pkl"
joblib.dump(soil_model_payload, model_filename)
print(f"[+] Production pipeline asset saved successfully as: '{model_filename}'\n")


def real_time_soil_inference(sensor_dict, model_path="unsupervised_soil_pipeline.pkl"):
    """
    Accepts raw sensor readings dictionary, applies training preprocessing parameters,
    and calculates an automated soil safety viability recommendation.
    """
    try:
        # Load local production pipeline assets
        pipeline = joblib.load(model_path)
        
        # Enforce accurate feature schema alignment
        ordered_features = ["Nitrogen", "Phosphorus", "Potassium", "pH", "EC"]
        input_df = pd.DataFrame([sensor_dict])[ordered_features]
        
        # Run transformation transformations
        scaled_input = pipeline["scaler_transform"].transform(input_df)
        
        # Predict mathematical cluster index
        target_cluster = pipeline["clustering_engine"].predict(scaled_input)[0]
        
        # Translate cluster index to real-world recommendation label
        final_verdict = pipeline["label_map"][target_cluster]
        return f"Prediction System Verdict: {final_verdict}"
        
    except Exception as e:
        return f"Error executing inference sequence: {str(e)}"

# --- Scenario Testing ---
# Simulate a live sensor configuration with highly balanced parameters
test_telemetry_sample = {
    "Nitrogen": 145.2,
    "Phosphorus": 52.1,
    "Potassium": 190.4,
    "pH": 6.62,
    "EC": 1.15
}

print("--- Running Test Inference Sequence ---")
print(real_time_soil_inference(test_telemetry_sample))

## 8. Automated Soil Remediation Advisory System
*Translates unsupervised cluster diagnostics into custom, actionable prescriptive advice. This component scans unviable soil profiles and dynamically generates targeted structural and chemical remediation strategies to restore land balance.*

In [ ]:
def generate_unsupervised_prescription(row):
    """
    Analyzes an unsupervised soil sample row and returns targeted agronomic advice.
    """
    prescription = []
    
    # 1. Address Extreme pH Displacements (As caught by the Red Cluster)
    pH = row['pH']
    if pH < 6.0:
        prescription.append(
            f"⚠️ HIGH ACIDITY (pH: {pH:.2f}): Soil matrix is too acidic. "
            "Apply Agricultural Lime (Calcium Carbonate) to unlock natural phosphorus bonds."
        )
    elif pH > 7.5:
        prescription.append(
            f"⚠️ HIGH ALKALINITY (pH: {pH:.2f}): Sodic/Calcareous soil detected. "
            "Incorporate Elemental Sulfur or Aluminum Sulfate to safely lower pH levels."
        )
        
    # 2. Address Severe Salinity Stress (The 16,000+ EC Outliers)
    ec = row['EC']
    if ec > 400:  # Scaled boundary for toxic salinity threshold
        prescription.append(
            f"🚨 CRITICAL SALINITY HAZARD (EC: {ec:.2f}): Severe salt accumulation. "
            "Initiate freshwater leaching/flushing and improve drainage to protect crop root osmolarity."
        )
        
    # 3. Address Nitrogen Deficiencies
    n = row['Nitrogen']
    if n < 500:  # Threshold based on your log-scale distribution
        prescription.append(
            f"📉 NITROGEN DEFICIENCY (N: {n:.2f}): Depleted nitrogen pool. "
            "Incorporate a leguminous cover crop or apply calculated nitrogenous fertilizer."
        )
        
    # 4. Address Excessive Phosphorus Accumulation Runaway
    p = row['Phosphorus']
    if p > 150:
        prescription.append(
            f"⚠️ PHOSPHORUS ACCUMULATION (P: {p:.2f}): Hyper-saturated fertilizer buildup. "
            "Immediately halt NPK inputs and use deep-rooted catch crops to deplete soil P reserves."
        )

    # 5. Address Potassium Imbalances
    k = row['Potassium']
    if k > 500:
        prescription.append(
            f"⚠️ POTASSIUM OVER-SATURATION (K: {k:.2f}): Mineral lockout risk. "
            "Check Magnesium levels to ensure high Potassium isn't inducing an Mg deficiency."
        )

    if not prescription:
        return ["✅ OPTIMAL SOIL BALANCE: Maintain current organic soil management practices."]
    
    return prescription


# =====================================================================
# AUTOMATED TEST: Run the advisory engine on your unviable cluster data
# =====================================================================
print("[*] Filtering out localized anomalies for prescriptive advisory sampling...\n")

# Isolate the data flagged as Unviable by your multi-criteria logic
unviable_samples = df[df['System_Recommendation'] == "Unviable / Deficient"]

if not unviable_samples.empty:
    # Take 3 random high-stress samples to audit how well the advisory engine works
    sample_size = min(3, len(unviable_samples))
    audit_subset = unviable_samples.sample(n=sample_size, random_state=42)
    
    for idx, (row_idx, row) in enumerate(audit_subset.iterrows(), 1):
        print(f"--- REMEDIATION PROFILE FOR ROW ID: {row_idx} ---")
        print(f"Raw Metrics -> N: {row['Nitrogen']:.1f} | P: {row['Phosphorus']:.1f} | K: {row['Potassium']:.1f} | pH: {row['pH']:.2f} | EC: {row['EC']:.1f}")
        
        prescriptions = generate_unsupervised_prescription(row)
        for step, advice in enumerate(prescriptions, 1):
            print(f"  {step}. {advice}")
        print("\n" + "="*80 + "\n")
else:
    print("[!] No unviable samples found in the current dataset slice to evaluate.")